# polygon → MOC → shard → 3-D → numpy

The whole zagg read stack in two libraries, one `%pip install`, and zero
credentials. A geojson polygon becomes a morton MOC; the MOC checks itself
against the store's own coverage; the covered shards open with timings; one
shard renders in 3-D (ATL03 + GEDI together); the current view exports to
voxel cubes on any grid you name and saves to disk.

Everything below is reader-side and calls no zagg public API — `mortie` for
the geometry, `moczarr` for the store, plus the t-digest algebra that
`moczarr[zagg]` imports from zagg rather than vendoring (moczarr issue #19).
That algebra is the only zagg code on the path. It all runs anonymously
against public S3, binder-ready.

Its sibling is [`waveform_viewer.ipynb`](waveform_viewer.ipynb), which takes
the same polygon and stores down to the cell-level join: one GEDI o18
footprint against the 2×2 ATL03 o19 cells beneath it, both rebuilt from their
stored t-digests. The two are separate notebooks because this one needs
`%matplotlib widget` and that one `%matplotlib inline`; the backends collide
in a single kernel.

In [ ]:
%pip install -q mortie "moczarr[zagg]>=0.7" matplotlib ipympl ipywidgets
%matplotlib widget

import json
import os
import time
import zipfile
from io import BytesIO

import moczarr as mz
import numpy as np
from moczarr.hhdc import (
    block_rank,
    chunk_z_range,
    rank_to_rowcol,
    rasterize_cell,
    rowcol_to_rank,
)
from mortie import generate_morton_children, moc

# The drawing lives in viewers.py beside this notebook, so the cells below stay
# about the READ path. Both demo notebooks share it.
from viewers import (
    BLOCK_ORDER,
    SIDE,
    UNITS,
    block_of,
    densest_shard,
    human_bytes,
    joint_cells,
    view3d,
)

# One store per product, each appendable. Coverage answers which ground the
# store holds; the store name never does.
STORES = {
    "atl03": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr",
        "19/h_tdigest_signal",
    ),
    "gedi": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/gedi_flux_o9.zarr",
        "18/rx_flux",
    ),
}
S3 = {"region": "us-west-2", "anonymous": True}

## One polygon in, covered shards out

The polygon below sits on SERC, the mid-Atlantic forest site the HHDC
diffusion papers are built on. Replace it with any area within California or a
NEON AOP site — it need not be a box — and the cell tests whether each store
actually covers what you asked for. An AOI in Alaska will pass
the ATL03 check and fail the GEDI one — GEDI flies on the ISS, so it sees no
higher than |lat| 51.6 and the Alaska NEON sites are outside its reach.

In [ ]:
# ~7 km2 over the Smithsonian Environmental Research Center, Edgewater MD --
# the site the HHDC diffusion literature is built on. Mkaouar et al. state it
# at 38°53'33"N 76°33'37"W and simulate a 600 x 600 m DART scene there; the
# same tract carries the real GEDI footprints they match against (orbits 02273
# and 19178), and it is NEON AOP site SERC. Closed-canopy mid-Atlantic
# deciduous -- sweetgum and 30-45 m tulip poplar over dense ironwood.
#
# Not a box. The southwest corner is cut back because a full rectangle runs
# into the open water of the Rhode River, which returns no canopy at all.
aoi = {
    "features": [
        {
            "geometry": {
                "coordinates": [
                    [
                        [-76.5750, 38.9000],
                        [-76.5450, 38.9000],
                        [-76.5450, 38.8780],
                        [-76.5600, 38.8720],
                        [-76.5750, 38.8800],
                        [-76.5750, 38.9000],
                    ]
                ]
            }
        }
    ]
}

q = moc(aoi)
shards = None
for name, (root, _field) in STORES.items():
    assert mz.coverage_moc(root, **S3).contains(q), f"{name} does not contain the polygon"
    ids = set(mz.candidate_shards(root, aoi=q, **S3))
    shards = ids if shards is None else shards & ids
shards = sorted(shards)

# Everything below works ONE shard, named here and only here. Change the index
# to look at another -- opening one shard and viewing another leaves every read
# asking for a subtree outside the open leaf's axis, which comes back as a
# warning and an empty pane rather than an error.
# Same rule demo/06_paired used: the shard with the most GEDI granules, so
# both demos land on the same one. `read_stats` reads each leaf's telemetry
# sidecar by path arithmetic -- a few KB, no listing.
SHARD = densest_shard(STORES["gedi"][0], shards, **S3)
print(f"{len(shards)} shards cover the polygon: {shards}\nworking {SHARD}")

## Open one shard — every dataset, timed

In [ ]:
def open_shard(shard):
    """Open each store's leaf for this shard, and price a sweep of ONE field.

    The sweep decodes every stored digest of the named field across all 64
    blocks of the shard -- so the seconds and megabytes below are the cost of
    reading one ARRAY end to end, not of reading the leaf. This ATL03 leaf holds
    nine arrays (46.7 MiB on S3): the signal digests read here, their `locations`
    and `times` companions, the whole `h_tdigest_noise` channel with companions
    of its own, plus `morton`, `composition` and `count`. Reading the leaf costs
    several times what this prints.

    Nor is it every photon: signal only. On this shard `19/count` sums to
    3,010,061, which is the 1,266,765 signal centroids plus 1,743,296 noise ones.

    None of what it decodes is kept -- the viewer below fetches one block at a
    time, which is what holds this notebook inside Binder's 2 GB when the sparse
    stored digests are cast to the dense tensors visualization needs. The
    per-block cell and observation counts fall out of the same pass for free, so
    they come back too and label the viewer's block dropdown -- along with the
    count of cells BOTH sensors populate, which is what that dropdown is ordered
    by. They are STORED totals; the viewer's own per-read counts are clipped to a
    finite z window and run lower.
    """
    handles, tally, words = {}, {}, {}
    for name, (root, field) in STORES.items():
        t0 = time.perf_counter()
        store = mz.open_leaf(root, shard, **S3)
        _, element = mz.open_ragged(store, field)
        per, cells, centroids, nbytes, obs = {}, 0, 0, 0, 0.0
        seen = []
        for word, value in mz.read_ragged(store, field):
            v = np.asarray(value)
            seen.append(word)
            weight = float(v[:, 1].sum())  # column 1 is the centroid weight
            block = int(block_of(word, BLOCK_ORDER))
            had_cells, had_obs = per.get(block, (0, 0.0))
            per[block] = (had_cells + 1, had_obs + weight)
            cells, centroids, nbytes, obs = (
                cells + 1,
                centroids + len(v),
                nbytes + v.nbytes,
                obs + weight,
            )
        tally[name] = per
        words[name] = np.asarray(seen, dtype=np.uint64)
        print(
            f"{name:6s} swept {field} in {time.perf_counter() - t0:5.1f}s — "
            f"{cells:,} cells over {len(per)} blocks, {centroids:,} centroids, "
            f"{obs:,.0f} {UNITS[name]}, {nbytes / 2**20:.1f} MiB decoded "
            f"(this ONE array, not the whole leaf)"
        )
        print(f"       element {element}")
        handles[name] = (store, field)
    return handles, tally, joint_cells(words)


handles, tally, joint = open_shard(SHARD)


## The 3-D view — both sensors, exact centroids, time-aware

In [ ]:
view = view3d(handles, SHARD, tally=tally, joint=joint)  # ordered by coincident cells


## Export — voxel cubes, on a grid you choose

Both exports leave `read_tensors` and build from the digests directly, because
each needs something it cannot give.

**1 — ATL03 alone, finer than the store.** The located companion carries one
order-29 point word per centroid, so the cube can be finer than the o19 cells
the digests are keyed by. Default o22: 1.554 m voxels, z binned to match, the
block emitted as 8×8 isotropic 128³ chips (199 m a side), empty chips skipped.
`order=24` gives 0.389 m voxels in 49.7 m chips — many more, mostly empty.

**2 — ATL03 and GEDI co-registered**, ready to stack: one xy lattice (ATL03's
o19, each GEDI o18 cell replicated into its four children) and one z axis for
both. The z axis is the part worth watching — `read_tensors` derives its window
per sensor, which on one block here is `z0 = -71.0` for ATL03 against `-59.0`
for GEDI, twelve bins out of register.

In [ ]:
def voxel_chips(block, sensor="atl03", order=22, side=128, n_bins=128):
    """Located centroids -> isotropic `side`**3 count chips, empty ones skipped.

    Shifting a centroid's block-local rank right by `2 * (29 - order)` truncates
    its order-29 point word to `order` -- nested ranks are hierarchical -- which
    is how the cube gets finer than the cells. The z bin equals the cell edge,
    so voxels are cubes; each chip keeps its own `z0` in `meta.json`.
    """
    store, field = handles[sensor]
    dz = SIDE / 2 ** (order - BLOCK_ORDER)
    depth = order - (side.bit_length() - 1) - BLOCK_ORDER
    tiles = generate_morton_children(int(block), order - (side.bit_length() - 1))

    t0 = time.perf_counter()
    got = list(mz.read_ragged(store, field, locations=True, subtree=mz.morton_decimal(int(block))))
    v = np.concatenate([np.asarray(r[1]) for r in got])
    z, wt = v[:, 0], v[:, 1]
    rank = block_rank(np.concatenate([np.asarray(r[2], np.uint64) for r in got]), BLOCK_ORDER)[0]
    row, col = rank_to_rowcol(rank >> np.uint64(2 * (29 - order)), order - BLOCK_ORDER)
    read_s = time.perf_counter() - t0

    # Stream each chip out and drop it: 64 x 8 MiB held at once is 512 MiB, and
    # mybinder caps the container at 2 GB. An .npz is a zip of .npy members.
    path = f"{sensor}_o{order}_chips_{mz.morton_decimal(int(block))}.npz"
    t1 = time.perf_counter()
    meta, kept, dense = {}, 0, 0
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for i in range(2**depth):
            for j in range(2**depth):
                m = np.flatnonzero((row // side == i) & (col // side == j))
                if not len(m):
                    continue
                z0 = np.floor(z[m].min())
                iz = ((z[m] - z0) / dz).astype(np.int64)
                m, iz = m[iz < n_bins], iz[iz < n_bins]  # relief past the window
                acc = np.zeros((side, side, n_bins), dtype=np.float32)
                np.add.at(acc, (row[m] % side, col[m] % side, iz), wt[m])
                chip = np.rint(acc).astype(np.uint32)  # merged centroids weigh fractionally
                name = mz.morton_decimal(int(tiles[rowcol_to_rank(i, j, depth=depth)]))
                buf = BytesIO()
                np.save(buf, chip)
                zf.writestr(f"{name}.npy", buf.getvalue())
                meta[name] = {"z0": float(z0), "dz": dz, "written": int(chip.sum())}
                kept += int(chip.sum())
                dense += chip.nbytes
        zf.writestr("meta.json", json.dumps(meta))  # z0 per chip; absolute z is recoverable

    n, on_disk = 2 ** (2 * depth), os.path.getsize(path)
    print(f"{sensor} o{order} — {dz:.3f} m isotropic voxels, {side}^3 = {dz * side:.0f} m chips")
    print(f"  read   {len(z):,} centroids, {wt.sum():,.0f} {UNITS[sensor]}, {read_s:.1f}s")
    print(f"  wrote  {len(meta)} of {n} chips, {kept:,} of {wt.sum():,.0f} {UNITS[sensor]} "
          f"({wt.sum() - kept:,.0f} outside a chip's z window), {time.perf_counter() - t1:.1f}s")
    print(f"         {human_bytes(dense)} dense -> {human_bytes(on_disk)} ({dense / on_disk:.0f}x)")
    return path, meta


In [ ]:
chips, manifest = voxel_chips(view.block)  # the block on screen, at o22
# chips24, _ = voxel_chips(view.block, order=24)  # 0.389 m voxels, 1,024 chips


In [ ]:
def registered_pair(block, order=19, n_bins=128, resolution=0.5):
    """Both sensors as cubes of ONE shape: one xy lattice, one z axis.

    Each GEDI o18 cell is REPLICATED into its four o19 children rather than
    ATL03 being merged up to o18 -- so a GEDI cube sums to four times its
    stored weight. `chunk_z_range` is handed both sensors' digests together;
    derived per sensor it puts them bins apart, and nothing downstream notices.
    """
    t0 = time.perf_counter()
    side = 2 ** (order - BLOCK_ORDER)
    got = {
        n: list(mz.read_ragged(store, field, subtree=mz.morton_decimal(int(block))))
        for n, (store, field) in handles.items()
    }
    read_s = time.perf_counter() - t0

    z0, n_bins, dz = chunk_z_range(
        [np.asarray(v) for rows in got.values() for _w, v in rows],
        n_bins=n_bins, resolution=resolution, bottom=0.05, top=0.95, fit="degrade_resolution",
    )
    t1 = time.perf_counter()
    cubes, lines = {}, []
    for name, (_store, field) in handles.items():
        cell_order = int(field.split("/", 1)[0])
        k = 2 ** (order - cell_order)  # children of one cell on the output grid
        words = np.array([w for w, _v in got[name]], dtype=np.uint64)
        r, c = rank_to_rowcol(block_rank(words, BLOCK_ORDER)[0], cell_order - BLOCK_ORDER)
        cube = np.zeros((side, side, n_bins), dtype=np.float32)
        for i, (_w, v) in enumerate(got[name]):
            cube[r[i] * k : (r[i] + 1) * k, c[i] * k : (c[i] + 1) * k] = rasterize_cell(
                np.asarray(v), z0, dz, n_bins
            )
        cubes[name] = cube
        lines.append(
            f"         {name}: {len(words):,} o{cell_order} cells -> {k}x{k} -> "
            f"{int((cube.sum(2) > 0).sum()):,}/{side * side:,} columns, "
            f"{cube.sum():,.0f} {UNITS[name]}" + (f" ({k**2}x replicated)" if k > 1 else "")
        )

    path = f"registered_o{order}_{mz.morton_decimal(int(block))}.npz"
    np.savez_compressed(path, **cubes, z0=z0, dz=dz, order=order)
    dense, on_disk = sum(c.nbytes for c in cubes.values()), os.path.getsize(path)
    fit = "as asked" if abs(dz - resolution) < 1e-9 else f"DEGRADED from {resolution:g} m"
    print(f"registered o{order} — {next(iter(cubes.values())).shape} float32 each, {read_s:.1f}s read")
    print(f"  grid   shared z = {z0:.1f} m + bin * {dz:g} m ({fit})")
    print("\n".join(lines))
    print(f"  wrote  {human_bytes(dense)} dense -> {human_bytes(on_disk)} in {path}, "
          f"{time.perf_counter() - t1:.1f}s")
    return path, cubes


In [ ]:
pair, cubes = registered_pair(view.block)
stacked = np.stack([cubes["atl03"], cubes["gedi"]], axis=0)  # registered, so they stack
print(f"stacked {stacked.shape} — {human_bytes(stacked.nbytes)}, ready for a 2-channel model")


Two libraries, one polygon — coverage, shards, timings, the paired 3-D view,
and co-registered voxel cubes on disk.